# 0. Initialisation et configuration

In [1]:
pip install linearmodels


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from linearmodels.iv import IV2SLS

# --- CONFIGURATION (La seule partie à modifier si la base change) ---
CONFIG = {
    "file_name": "../données/OECD_debt_recent.csv",
    "mapping": {
        "Pays": "country",
        "Year": "year",
        "Nominal gross domestic product (billions)": "gdp",
        "Household debt, loans and debt securities (percent of GDP)": "debt_hh",
        "Non-financial corporations debt, loans and debt securities (percent of GDP)": "debt_corp",
        "General government debt (percent of GDP)": "debt_gov"
    },
    "target": "growth",       # Nom générique de la variable à prédire
    "interest_vars": ["debt_hh", "debt_corp", "debt_gov"] # Nos variables de dette
}

## Nettoyage

In [3]:

def load_and_prepare_data(config):
    # Chargement
    df = pd.read_csv(config["file_name"])
    
    # Renommage selon le mapping
    df = df.rename(columns=config["mapping"])
    
    # Filtrage : on enlève les colonnes non renseignées (all instruments)
    keep_cols = list(config["mapping"].values())
    df = df[keep_cols]
    
    # Nettoyage des types (Conversion string "12,5" -> float 12.5)
    for col in df.columns:
        if col not in ["country", "year"]:
            df[col] = df[col].astype(str).str.replace(',', '.').replace('nan', np.nan).astype(float)
    
    # Tri chronologique par pays
    df = df.sort_values(["country", "year"])
        
    return df

# Exécution
df_master = load_and_prepare_data(CONFIG)
print(f"Base de données prête : {df_master.shape[0]} lignes, {df_master.shape[1]} colonnes.")
print(df_master.head())

Base de données prête : 1102 lignes, 6 colonnes.
     country  year     gdp  debt_hh  debt_corp  debt_gov
0  Australia  1996  542.40    56.57      59.33     29.35
1  Australia  1997  573.07    59.98      61.28     25.91
2  Australia  1998  606.13    63.55      62.99     23.70
3  Australia  1999  638.38    67.93      63.90     22.54
4  Australia  2000  687.43    70.18      68.72     19.50


In [4]:
df_master.head()

,country,year,gdp,debt_hh,debt_corp,debt_gov
0,Australia,1996,542.40,56.57,59.33,29.35
1,Australia,1997,573.07,59.98,61.28,25.91
2,Australia,1998,606.13,63.55,62.99,23.70
3,Australia,1999,638.38,67.93,63.90,22.54
4,Australia,2000,687.43,70.18,68.72,19.50


In [5]:
df_master.isna().sum().sum()

145

Il y a 145 NaN dans notre data frame.

In [6]:
df_master.isna().sum(axis=1).value_counts().sort_index()


0    1001
1      66
2      26
3       9
Name: count, dtype: int64

1001 lignes n'ont pas de NaN sur les 1101 lignes de notre data frame. Cela représente 91% du data frame. 

In [7]:
df_master.head(100)

,country,year,gdp,debt_hh,debt_corp,debt_gov
0,Australia,1996,542.40,56.57,59.33,29.35
1,Australia,1997,573.07,59.98,61.28,25.91
2,Australia,1998,606.13,63.55,62.99,23.70
3,Australia,1999,638.38,67.93,63.90,22.54
4,Australia,2000,687.43,70.18,68.72,19.50
...,...,...,...,...,...,...
95,Canada,2004,1335.73,69.61,84.13,71.89
96,Canada,2005,1421.59,72.26,77.42,70.64
97,Canada,2006,1496.60,75.90,78.26,69.92
98,Canada,2007,1577.66,80.73,81.12,67.18


Pour remplir les Nan, on va utiliser la méthode d'interpolation qui remplit les NaN par une estimation linéaire de ce que devrait être la valeur étant donné son passé et son future. 

In [8]:
cols_to_interp = ["gdp", "debt_hh", "debt_corp", "debt_gov"]

df_master = df_master.sort_values(["country", "year"])

# CORRECTION : on distingue deux étapes pour ne pas fabriquer de données.
#
# Étape 1 — interpolation LINÉAIRE interne uniquement (limit_direction='forward')
#   → comble les NaN situés ENTRE deux observations valides.
#   → ne touche PAS aux NaN de tête (avant la 1ère valeur valide d'un pays).
# Étape 2 — bfill(limit=2) pour les NaN de tête uniquement
#   → recopie au maximum 2 périodes vers l'arrière en début de série.
#   → limit=2 évite de propager une unique valeur de référence sur toute
#     une séquence initiale longue (extrapolation déguisée).
# Les NaN résiduels (pays avec moins de 2 observations) seront éliminés
# par dropna() lors de la construction du panel.

df_master[cols_to_interp] = (
    df_master
        .groupby("country")[cols_to_interp]
        .transform(lambda x: x.interpolate(method="linear", limit_direction="forward")
                               .bfill(limit=2))
)

print(f"NaN restants après interpolation : {df_master[cols_to_interp].isna().sum().sum()}")
print("(Les NaN résiduels concernent des pays sans aucune donnée sur une variable ;")
print(" ils seront supprimés automatiquement par dropna() dans les modèles.)")

NaN restants après interpolation : 119
(Les NaN résiduels concernent des pays sans aucune donnée sur une variable ;
 ils seront supprimés automatiquement par dropna() dans les modèles.)


In [9]:
df_master.isna().sum().sum()

119

L'interpolation se fait en deux étapes :
1. **Interpolation linéaire interne** (`limit_direction='forward'`) : comble les trous situés *entre* deux observations valides sans extrapoler.
2. **`bfill(limit=2)`** : recopie au maximum 2 périodes vers l'arrière pour traiter les NaN en début de série de certains pays.

Cette approche est plus conservative que `limit_direction='both'` (version précédente) qui extrapolait aussi en dehors des bornes.

In [10]:
# Création des Lags (Dette à t-1)
interest_vars = ["debt_hh", "debt_corp", "debt_gov"]

for var in interest_vars:
    df_master[f"{var}_lag1"] = df_master.groupby("country")[var].shift(1)

# ⚠ AVERTISSEMENT — PIB NOMINAL
# La variable 'gdp' est le PIB nominal en milliards de monnaie nationale.
# Le taux de croissance calculé ci-dessous est donc un taux de croissance
# NOMINAL : il capture à la fois la croissance réelle et l'inflation.
#
# Idéalement, il faudrait diviser par un déflateur du PIB pour obtenir
# la croissance en volume. En l'absence de déflateur dans la base,
# deux éléments atténuent partiellement ce biais :
#   - Les effets fixes par pays (alpha_i) absorbent les niveaux d'inflation
#     structurellement différents entre pays.
#   - Les CS-means (moyennes transversales) captent les chocs d'inflation
#     communs (ex. chocs pétroliers, épisodes de désinflation mondiale).
# Cette limitation doit être mentionnée comme caveat dans l'analyse.

df_master["growth"] = (
    df_master
        .groupby("country")["gdp"]
        .apply(lambda x: np.log(x).diff() * 100)
        .reset_index(level=0, drop=True)
)

print("Variable 'growth' créée (croissance nominale du PIB, en %).")
print(f"Moyenne : {df_master['growth'].mean():.2f}%  |  Std : {df_master['growth'].std():.2f}%")

Variable 'growth' créée (croissance nominale du PIB, en %).
Moyenne : 6.07%  |  Std : 6.81%


In [11]:
df_master.head(20)

,country,year,gdp,debt_hh,debt_corp,debt_gov,debt_hh_lag1,debt_corp_lag1,debt_gov_lag1,growth
0,Australia,1996,542.40,56.57,59.33,29.35,NaN,NaN,NaN,NaN
1,Australia,1997,573.07,59.98,61.28,25.91,56.57,59.33,29.35,5.500414
2,Australia,1998,606.13,63.55,62.99,23.70,59.98,61.28,25.91,5.608661
3,Australia,1999,638.38,67.93,63.90,22.54,63.55,62.99,23.70,5.183923
4,Australia,2000,687.43,70.18,68.72,19.50,67.93,63.90,22.54,7.402629
5,Australia,2001,730.49,73.88,65.36,17.11,70.18,68.72,19.50,6.075554
6,Australia,2002,782.37,81.39,63.69,15.01,73.88,65.36,17.11,6.861223
7,Australia,2003,830.19,90.62,62.04,13.19,81.39,63.69,15.01,5.932682
8,Australia,2004,894.33,96.79,61.93,11.91,90.62,62.04,13.19,7.442024
9,Australia,2005,964.21,101.33,67.78,10.86,96.79,61.93,11.91,7.523428


In [12]:
df_master.to_csv("dette_master.csv", index=False, encoding="utf-8-sig")

## Test de stationnarité (Im-Pesaran-Shin)

Avant d'estimer les modèles, on vérifie que nos variables de panel ne sont pas intégrées d'ordre 1 (I(1)), 
ce qui invaliderait les régressions en niveaux.

On utilise le test **IPS (Im-Pesaran-Shin, 2003)** qui est adapté aux panels hétérogènes :
- Pour chaque pays $i$, on estime un test ADF individuel et on récupère la statistique $t_i$.
- La statistique IPS est la moyenne standardisée des $t_i$ : $W = \sqrt{N} \cdot (\bar{t} - E[t_i]) / \sqrt{Var(t_i)}$
- Sous H₀ : toutes les séries sont I(1). Sous H₁ : une fraction des séries est I(0).

> Les valeurs critiques et les moments $E[t_i]$, $Var(t_i)$ sont tirés de la Table 3 de Im, Pesaran et Shin (2003) 
pour le cas avec constante, sans tendance (cas le plus adapté à nos variables en % du PIB).

In [13]:
from statsmodels.tsa.stattools import adfuller

# Moments IPS Table 3 — cas avec constante, sans tendance
# Indexés par T (longueur de la série individuelle)
# Source : Im, Pesaran & Shin (2003), Journal of Econometrics
IPS_MOMENTS = {
    # T : (E[t_bar], Var[t_bar])
    10: (-1.520, 1.045), 15: (-1.620, 0.897), 20: (-1.659, 0.851),
    25: (-1.676, 0.830), 30: (-1.685, 0.820), 40: (-1.693, 0.813),
    50: (-1.697, 0.810), 100: (-1.702, 0.806),
}

def get_ips_moments(T):
    """Interpolation linéaire des moments IPS pour un T donné."""
    keys = sorted(IPS_MOMENTS.keys())
    if T <= keys[0]:  return IPS_MOMENTS[keys[0]]
    if T >= keys[-1]: return IPS_MOMENTS[keys[-1]]
    for i in range(len(keys)-1):
        if keys[i] <= T <= keys[i+1]:
            t0, t1 = keys[i], keys[i+1]
            w = (T - t0) / (t1 - t0)
            e = IPS_MOMENTS[t0][0] + w * (IPS_MOMENTS[t1][0] - IPS_MOMENTS[t0][0])
            v = IPS_MOMENTS[t0][1] + w * (IPS_MOMENTS[t1][1] - IPS_MOMENTS[t0][1])
            return e, v


def ips_test(df, var, max_lags=2):
    """
    Calcule la statistique IPS W-bar pour la variable 'var'.
    Retourne : (W_stat, p_approx, t_stats individuels)
    """
    t_stats, N_valid = [], 0
    for country, grp in df.groupby('country'):
        series = grp.sort_values('year')[var].dropna()
        if len(series) < 8:
            continue
        try:
            adf_result = adfuller(series, maxlag=max_lags, regression='c', autolag='AIC')
            t_stats.append(adf_result[0])
            N_valid += 1
        except Exception:
            continue

    if N_valid < 3:
        return np.nan, np.nan, []

    t_bar = np.mean(t_stats)
    T_avg = int(df.groupby('country')[var].count().mean())
    E_t, Var_t = get_ips_moments(T_avg)

    W = np.sqrt(N_valid) * (t_bar - E_t) / np.sqrt(Var_t)

    # p-value approchée : W ~ N(0,1) sous H0
    from scipy.stats import norm
    p_val = norm.cdf(W)   # test unilatéral à gauche

    return W, p_val, t_stats


vars_to_test = ['growth', 'debt_hh', 'debt_corp', 'debt_gov']
print('Test IPS (Im-Pesaran-Shin, 2003)')
print('H0 : toutes les séries sont I(1)  |  H1 : fraction des séries est I(0)')
print('Valeurs critiques (unilatéral gauche) : -1.65 (5%), -1.28 (10%)')
print('=' * 65)
print(f"  {'Variable':<15} {'W-stat':>8} {'p-value':>9} {'Conclusion':>20}")
print('-' * 65)

ips_results = {}
for var in vars_to_test:
    W, p, t_indiv = ips_test(df_master, var)
    ips_results[var] = (W, p)
    if np.isnan(W):
        concl = 'données insuffisantes'
    elif p < 0.05:
        concl = 'Rejette H0 → I(0)'
    elif p < 0.10:
        concl = 'Rejette H0 * → I(0)'
    else:
        concl = 'Ne rejette pas H0'
    p_str = f"{p:.4f}" if not np.isnan(p) else '—'
    W_str = f"{W:.4f}" if not np.isnan(W) else '—'
    print(f"  {var:<15} {W_str:>8} {p_str:>9} {concl:>20}")

print('=' * 65)
print("\nNote : si une variable de dette n'est pas I(0), son inclusion"
      " en niveau reste valide dans le CS-ARDL car les CS-means\n"
      "       absorbent la tendance commune stochastique (Pesaran, 2007).")

Test IPS (Im-Pesaran-Shin, 2003)
H0 : toutes les séries sont I(1)  |  H1 : fraction des séries est I(0)
Valeurs critiques (unilatéral gauche) : -1.65 (5%), -1.28 (10%)
  Variable          W-stat   p-value           Conclusion
-----------------------------------------------------------------
  growth          -15.9418    0.0000    Rejette H0 → I(0)
  debt_hh           0.2989    0.6175    Ne rejette pas H0
  debt_corp        -1.8590    0.0315    Rejette H0 → I(0)
  debt_gov          2.7227    0.9968    Ne rejette pas H0

Note : si une variable de dette n'est pas I(0), son inclusion en niveau reste valide dans le CS-ARDL car les CS-means
       absorbent la tendance commune stochastique (Pesaran, 2007).


# Premières régressions

## OLS naïf

In [14]:
# On retire les lignes vides (dues au calcul des lags et de la croissance)
df_model = df_master.dropna(subset=["growth"] + [v+"_lag1" for v in CONFIG["interest_vars"]])

# Définition de la formule de manière dynamique
# Elle ressemblera à : "growth ~ debt_hh_lag1 + debt_corp_lag1 + debt_gov_lag1"
formula = "growth ~ " + " + ".join([f"{v}_lag1" for v in CONFIG["interest_vars"]])

print(f"Lancement du modèle : {formula}")

# Modèle OLS simple (Pooling)
model_ols = smf.ols(formula, data=df_model).fit()

# Affichage des résultats
print(model_ols.summary())

Lancement du modèle : growth ~ debt_hh_lag1 + debt_corp_lag1 + debt_gov_lag1
                            OLS Regression Results                            
Dep. Variable:                 growth   R-squared:                       0.130
Model:                            OLS   Adj. R-squared:                  0.127
Method:                 Least Squares   F-statistic:                     48.26
Date:                Sun, 22 Mar 2026   Prob (F-statistic):           4.43e-29
Time:                        17:39:15   Log-Likelihood:                -3052.7
No. Observations:                 976   AIC:                             6113.
Df Residuals:                     972   BIC:                             6133.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------

Ce modèle montre que la dette des ménages et celle de l'État pèsent significativement sur la croissance future, avec un impact négatif plus marqué pour les ménages ($-0,05$), tandis que la dette des entreprises n'a pas d'effet statistiquement prouvé ($p=0,44$). Malgré un $R^2$ de $12,8\%$, la faible valeur de Durbin-Watson ($1,09$) révèle une autocorrélation des résidus, indiquant que le modèle ignore des spécificités nationales ou des chocs temporels que les effets fixes et le modèle CS-ARDL devront traiter.

### Introduction des effets fixes

In [15]:
# Préparation des données pour le Panel (Index double : Pays puis Année)
df_panel = df_model.set_index(['country', 'year'])

# Modèle avec Effets Fixes par Pays (Entity Effects)
# On utilise les mêmes variables mais on ajoute "EntityEffects"
exog_vars = [f"{v}_lag1" for v in CONFIG["interest_vars"]]
exog = sm.add_constant(df_panel[exog_vars])

model_fe = PanelOLS(df_panel['growth'], exog, entity_effects=True)
results_fe = model_fe.fit()

print(results_fe)

                          PanelOLS Estimation Summary                           
Dep. Variable:                 growth   R-squared:                        0.0586
Estimator:                   PanelOLS   R-squared (Between):              0.0157
No. Observations:                 976   R-squared (Within):               0.0586
Date:                Sun, Mar 22 2026   R-squared (Overall):              0.0110
Time:                        17:39:15   Log-likelihood                   -2931.9
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      19.456
Entities:                          36   P-value                           0.0000
Avg Obs:                       27.111   Distribution:                   F(3,937)
Min Obs:                       16.000                                           
Max Obs:                       28.000   F-statistic (robust):             19.456
                            

Formellement, le modèle devient :

$$
growth_{it} = \alpha_i + \beta_1 \, debt\_hh\_lag1_{it} + \beta_2 \, debt\_corp\_lag1_{it} + \dots + \varepsilon_{it}
$$

où :

- $\alpha_i$ = **effet fixe du pays $i$**  
- Les coefficients $\beta$ sont estimés en utilisant la **variation dans le temps au sein de chaque pays**  
- Les différences **entre pays** sont absorbées par $\alpha_i$



L'introduction des effets fixes renforce l'impact négatif de la dette des ménages, dont le coefficient chute de $-0,05$ à $-0,09$ ($p=0,000$). Ce modèle explique $6\%$ de la variance intra-pays ($R^2$ Within), prouvant que les caractéristiques structurelles propres à chaque nation masquaient une partie de l'effet néfaste de l'endettement sur la croissance. La forte significativité des effets d'entité ($p=0,000$) valide cette approche par rapport à l'OLS simple et prépare le terrain pour le modèle CS-ARDL.

## Variables instrumentales

Afin de traiter un potentiel biais d'endogénéité nous utilisons la méthode des doubles moindres carrés (2SLS). Nous instrumentons la dette des ménages par son retard à deux périodes ($t-2$), partant du principe que le niveau d'endettement passé influence la croissance actuelle mais n'est pas directement impacté par les chocs économiques de l'année en cours. Cette approche permet d'isoler la variation de la dette qui est purement exogène afin de confirmer la direction du lien de causalité.

In [16]:
# Création des instruments : lag-2 de chaque variable de dette
# Hypothèse d'exclusion : le niveau d'endettement d'il y a 2 ans
# influence la croissance actuelle UNIQUEMENT via le niveau d'endettement
# d'il y a 1 an (canal), et non directement.
# Cette hypothèse est standard dans la littérature sur dette-croissance
# (voir Cecchetti et al., 2011 ; Lombardi et al., 2017).

for var in CONFIG["interest_vars"]:
    df_panel[f'{var}_lag2'] = df_panel.groupby('country')[var].shift(2)

df_iv = df_panel.dropna(
    subset=[f'{v}_lag2' for v in CONFIG["interest_vars"]] +
            [f'{v}_lag1' for v in CONFIG["interest_vars"]] +
            ['growth']
)

# --- Modèle IV pour chaque type de dette séparément ---
# On instrumente chaque dette_lag1 par son propre lag-2.
# Les autres dettes sont traitées comme exogènes (incluses en contrôle).

iv_results = {}
for target_var in CONFIG["interest_vars"]:
    other_vars = [f'{v}_lag1' for v in CONFIG["interest_vars"] if v != target_var]
    controls   = ' + '.join(other_vars)
    formula_iv = (f'growth ~ 1 + {controls} + '
                  f'[{target_var}_lag1 ~ {target_var}_lag2]')
    res_iv_tmp = IV2SLS.from_formula(formula_iv, data=df_iv).fit(cov_type='robust')
    iv_results[target_var] = res_iv_tmp
    print(f"\n{'='*60}")
    print(f"IV — variable instrumentée : {target_var}_lag1")
    print(f"{'='*60}")
    print(res_iv_tmp.summary)

# Pour la compatibilité avec le tableau comparatif, on garde
# le résultat de debt_hh dans results_iv (variable historique)
results_iv = iv_results['debt_hh']


IV — variable instrumentée : debt_hh_lag1
                          IV-2SLS Estimation Summary                          
Dep. Variable:                 growth   R-squared:                      0.1109
Estimator:                    IV-2SLS   Adj. R-squared:                 0.1079
No. Observations:                 904   F-statistic:                    83.319
Date:                Sun, Mar 22 2026   P-value (F-stat)                0.0000
Time:                        17:39:15   Distribution:                  chi2(3)
Cov. Estimator:                robust                                         
                                                                              
                               Parameter Estimates                                
                Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
----------------------------------------------------------------------------------
Intercept          9.5586     0.5968     16.018     0.0000      8.3890      

Le modèle IV est désormais estimé **séparément pour chaque type de dette**, 
en instrumentant `debt_X_lag1` par `debt_X_lag2`, tout en contrôlant pour les deux autres dettes (traitées comme exogènes).

Cette extension par rapport à la version précédente (qui n'instrumentait que `debt_hh`) 
permet de tester la robustesse au biais d'endogénéité pour l'ensemble des trois types d'endettement. 
La proximité des coefficients IV avec les coefficients OLS/FE suggère que le biais d'endogénéité 
est limité, ce qui renforce la crédibilité des résultats du CS-ARDL.

> **Hypothèse d'exclusion** : le lag-2 n'affecte la croissance courante qu'à travers le lag-1, 
et non directement. Cette hypothèse est standard dans la littérature (Cecchetti et al., 2011).

# Modèle CS-ARDL

Le modèle CS-ARDL constitue une extension dynamique du cadre de régression précédent. Il permet de répondre aux trois principaux problèmes identifiés dans la littérature empirique sur le lien entre dette et croissance, notamment par Lombardi (2012).

Premièrement, il introduit une dynamique temporelle via la structure Autoregressive Distributed Lag (ARDL). Contrairement à une régression statique, ce modèle inclut un retard de la variable dépendante ($y_{t-1}$) ainsi que des retards des variables explicatives ($x_{t-1}$). Cette spécification permet de capturer les mécanismes d’ajustement progressifs : l’effet de l’endettement sur la croissance ne se matérialise pas instantanément, mais se diffuse dans le temps.

Deuxièmement, l’approche ARDL permet de réduire les problèmes de simultanéité. En se fondant sur la chronologie des variables, le modèle exploite le fait que les niveaux d’endettement passés peuvent influencer la croissance future, alors que la croissance future ne peut pas affecter la dette passée. Cette structure dynamique limite ainsi les biais d’endogénéité présents dans les modèles statiques.

Troisièmement, le modèle corrige la dépendance transversale entre pays. Dans un panel d’économies de l’OCDE, les cycles conjoncturels sont fortement synchronisés. Des chocs globaux, tels que la crise financière de 2008, peuvent affecter simultanément la croissance de tous les pays. Sans correction, ces effets communs risquent d’être attribués à tort aux variables nationales d’endettement. Le modèle CS-ARDL introduit donc les moyennes transversales des variables (notées $\bar{Z}_t$), qui capturent ces facteurs globaux non observés.

La spécification estimée s’écrit :

$$y_{it} = \alpha_i + \sum_{j=1}^p \phi_j y_{i,t-j} + \sum_{j=0}^q \beta_j x_{i,t-j} + \sum_{j=0}^k \gamma_j \bar{y}_{t-j} + \sum_{j=0}^m \delta_j \bar{x}_{t-j} + \epsilon_{it}$$

où :

- $y_{i,t-j}$ est la croissance du pays $i$ à la date $t-j$,

- $x_{i,t-j}$ représente le vecteur des variables d’endettement nationales,

- $\bar{Z}_{t}$ correspond aux moyennes transversales des variables,

- $\alpha_i$ capte les caractéristiques structurelles propres à chaque pays.

- p, q, k et m le nombre de retards choisi

Cette spécification permet ainsi d’identifier un effet de la dette sur la croissance en tenant compte simultanément des dynamiques internes aux pays et des chocs macroéconomiques communs.

In [17]:
from linearmodels.panel import PanelOLS

def add_lags(df, group_col, vars_list, max_lag, suffix="lag"):
    out = df.copy()
    for v in vars_list:
        for L in range(1, max_lag+1):
            out[f"{v}_{suffix}{L}"] = out.groupby(group_col)[v].shift(L)
    return out

def add_cs_means(df, time_col, vars_list, lags_list=None, prefix="cs_"):
    out = df.copy()
    cols = vars_list.copy()
    if lags_list:
        cols += lags_list
    for c in cols:
        out[f"{prefix}{c}"] = out.groupby(time_col)[c].transform("mean")
    return out

# -------------------------
# PARAMS CS-ARDL
# -------------------------
q = 1  # nb de retards sur les dettes (à tester: 1,2,3)
p = 1  # nb de retards sur y (ici 1)

df_cs = df_master.copy()

# y = growth
# x = dettes en niveau (% GDP) plutôt que seulement lag1 déjà pré-calculé
x_vars = CONFIG["interest_vars"]  # ['debt_hh','debt_corp','debt_gov']

# Lags
df_cs = add_lags(df_cs, "country", ["growth"] + x_vars, max_lag=max(p, q), suffix="lag")

include_contemporaneous_x = True  # inclure les xit dans le modèle 

# Construire la liste des régressseurs
regressors = []
# AR part
regressors += [f"growth_lag{L}" for L in range(1, p+1)]

# DL part
if include_contemporaneous_x:
    regressors += x_vars
regressors += [f"{v}_lag{L}" for v in x_vars for L in range(1, q+1)]

# Ajouter CS means (contemporains + lags qui correspondent à ce que tu mets dans le modèle)
cs_source_cols = regressors  # on prend les mêmes colonnes → spec “symétrique”
df_cs = add_cs_means(df_cs, "year", vars_list=cs_source_cols, lags_list=None, prefix="cs_")

cs_regressors = [f"cs_{c}" for c in cs_source_cols]

# Dataset final
needed = ["country","year","growth"] + regressors + cs_regressors
df_cs = df_cs[needed].dropna().copy()
df_cs = df_cs.set_index(["country","year"])

# Formule PanelOLS
rhs = " + ".join(regressors + cs_regressors)
formula_cs_ardl = f"growth ~ {rhs} + EntityEffects"

model = PanelOLS.from_formula(formula_cs_ardl, data=df_cs, drop_absorbed=True)

res = model.fit(cov_type="clustered", cluster_entity=True)
print(res.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                 growth   R-squared:                        0.5750
Estimator:                   PanelOLS   R-squared (Between):             -15.332
No. Observations:                 945   R-squared (Within):               0.5750
Date:                Sun, Mar 22 2026   R-squared (Overall):             -10.578
Time:                        17:39:15   Log-likelihood                   -2458.6
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      86.493
Entities:                          36   P-value                           0.0000
Avg Obs:                       26.250   Distribution:                  F(14,895)
Min Obs:                       16.000                                           
Max Obs:                       27.000   F-statistic (robust):             47.168
                            

### Estimation des effets à long terme et à court terme

In [18]:
# 1. Extraction des coefficients
params = res.params

# 2. Calcul pour chaque variable de dette (x_vars)
lt_effects = {}
st_effects = {}

# On récupère le coefficient du retard de la variable dépendante (phi)
# Dans ton code p=1, donc c'est 'growth_lag1'
phi = params['growth_lag1']

for var in x_vars:
    # Effet Court Terme (CT) : coefficient de x_it
    beta_0 = params[var]
    st_effects[var] = beta_0
    
    # Effet Long Terme (LT) : (beta_0 + beta_1 + ...) / (1 - phi)
    # On récupère tous les retards de cette variable x (ici q=1)
    beta_lags = [params[f"{var}_lag{L}"] for L in range(1, q+1)]
    
    numerator = beta_0 + sum(beta_lags)
    denominator = 1 - phi
    
    lt_effects[var] = numerator / denominator

# 3. Affichage propre
results_df = pd.DataFrame({
    'Short-Term Effect': st_effects,
    'Long-Term Effect': lt_effects
})

print("--- Effets de la Dette sur la Croissance ---")
print(results_df)

--- Effets de la Dette sur la Croissance ---
           Short-Term Effect  Long-Term Effect
debt_hh            -0.328759         -0.010899
debt_corp           0.003048          0.003118
debt_gov           -0.197885         -0.035216


# Tests de robustesse et tableau comparatif

## Utilitaires partagés

On définit un wrapper `run_cs_ardl()` qui permet de relancer le modèle CS-ARDL 
sur n'importe quel sous-échantillon (groupe géographique ou sous-période), 
ainsi qu'une fonction `extract_effects()` pour en extraire les effets de court terme (CT) 
et de long terme (LT).

> **Note méthodologique** : les moyennes transversales (CS-means) sont recalculées 
à l'intérieur de chaque sous-groupe. C'est intentionnel : on veut que les facteurs 
communs capturés soient ceux propres au groupe étudié.

In [19]:
def run_cs_ardl(df_input, label='Full', p=1, q=1,
                x_vars_local=None, verbose=True):
    """
    Estime un CS-ARDL(p, q) sur le sous-échantillon df_input.

    Paramètres
    ----------
    df_input      : DataFrame avec colonnes country, year, growth, debt_*
    label         : nom du sous-échantillon (pour les prints)
    p             : ordre de retard sur y (growth)
    q             : ordre de retard sur les x (dettes)
    x_vars_local  : liste des variables de dette
    verbose       : affiche le résumé complet si True

    Retourne
    --------
    res           : objet résultat PanelOLS (ou None si données insuffisantes)
    """
    if x_vars_local is None:
        x_vars_local = ['debt_hh', 'debt_corp', 'debt_gov']

    df_cs = df_input.copy()

    # --- Lags sur y et les x ---
    df_cs = add_lags(df_cs, 'country', ['growth'] + x_vars_local,
                     max_lag=max(p, q), suffix='lag')

    # --- Régresseurs : AR(p) + DL(q) + contemporain ---
    regressors = [f'growth_lag{L}' for L in range(1, p + 1)]
    regressors += x_vars_local
    regressors += [f'{v}_lag{L}' for v in x_vars_local for L in range(1, q + 1)]

    # --- CS-means recalculées sur le sous-groupe ---
    df_cs = add_cs_means(df_cs, 'year', vars_list=regressors, prefix='cs_')
    cs_regressors = [f'cs_{c}' for c in regressors]

    # --- Nettoyage ---
    needed = ['country', 'year', 'growth'] + regressors + cs_regressors
    df_cs = df_cs[needed].dropna().copy()

    n_entities = df_cs['country'].nunique()
    n_obs      = len(df_cs)

    if n_entities < 5 or n_obs < 50:
        print(f'  [SKIP] "{label}" : {n_entities} pays, {n_obs} obs — sous-échantillon trop petit.')
        return None

    df_cs = df_cs.set_index(['country', 'year'])
    rhs   = ' + '.join(regressors + cs_regressors)
    model = PanelOLS.from_formula(f'growth ~ {rhs} + EntityEffects',
                                   data=df_cs, drop_absorbed=True)
    res   = model.fit(cov_type='clustered', cluster_entity=True)

    print(f"\n{'=' * 70}")
    print(f"  CS-ARDL({p},{q}) — Sous-échantillon : {label}")
    print(f"  Pays : {n_entities}  |  Observations : {n_obs}")
    print(f"{'=' * 70}")
    if verbose:
        print(res.summary)

    return res


def extract_effects(res, x_vars_local=None, q=1):
    """
    Calcule les effets Court Terme (CT) et Long Terme (LT).

    CT(x) = beta_0  (coeff de x_it)
    LT(x) = (beta_0 + beta_1 + ... + beta_q) / (1 - phi)
    """
    if x_vars_local is None:
        x_vars_local = ['debt_hh', 'debt_corp', 'debt_gov']

    if res is None:
        return ({v: float('nan') for v in x_vars_local},
                {v: float('nan') for v in x_vars_local})

    params = res.params
    phi    = params.get('growth_lag1', float('nan'))

    st_effects, lt_effects = {}, {}
    for var in x_vars_local:
        b0 = params.get(var, float('nan'))
        st_effects[var] = b0

        if np.isnan(b0) or np.isnan(phi) or phi >= 1:
            lt_effects[var] = float('nan')
        else:
            b_lags = [params.get(f'{var}_lag{L}', 0.0) for L in range(1, q + 1)]
            lt_effects[var] = (b0 + sum(b_lags)) / (1 - phi)

    return st_effects, lt_effects


def get_pval(res_obj, param_name):
    """Récupère la p-value d'un paramètre, retourne NaN si absent."""
    try:
        return float(res_obj.pvalues[param_name])
    except (KeyError, AttributeError):
        return float('nan')


def fmt(val, pval=None):
    """Formate un coefficient avec étoiles de significativité."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return '—'
    s = f'{val:.4f}'
    if pval is not None and not np.isnan(pval):
        if   pval < 0.01: s += '***'
        elif pval < 0.05: s += '**'
        elif pval < 0.10: s += '*'
    return s


x_vars = CONFIG['interest_vars']   # ['debt_hh', 'debt_corp', 'debt_gov']
print('Utilitaires chargés.')

Utilitaires chargés.


## A. Robustesse géographique : Europe vs OCDE hors Europe

On sépare le panel en deux groupes :
- **Europe** : pays membres de l'UE/EEE présents dans le panel ;
- **OCDE hors Europe** : États-Unis, Japon, Canada, Australie, Corée, etc.

L'objectif est de vérifier si les coefficients estimés sur l'ensemble du panel 
sont stables ou si l'effet de la dette sur la croissance diffère structurellement 
selon le contexte institutionnel.

In [20]:
# --- Liste des pays européens (noms tels qu'ils apparaissent dans l'OCDE) ---
EUROPE_COUNTRIES = [
    'Austria', 'Belgium', 'Czech Republic', 'Denmark', 'Estonia',
    'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Iceland',
    'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg',
    'Netherlands', 'Norway', 'Poland', 'Portugal', 'Slovak Republic',
    'Slovenia', 'Spain', 'Sweden', 'Switzerland', 'United Kingdom'
]

# Vérification dynamique contre les pays effectivement dans le panel
pays_in_data    = set(df_master['country'].unique())
europe_in_data  = [p for p in EUROPE_COUNTRIES if p in pays_in_data]
non_europe_in_data = sorted([p for p in pays_in_data if p not in EUROPE_COUNTRIES])

print('Pays EUROPÉENS reconnus dans le panel :', europe_in_data)
print('\nPays OCDE hors Europe :', non_europe_in_data)

df_europe     = df_master[df_master['country'].isin(europe_in_data)].copy()
df_non_europe = df_master[df_master['country'].isin(non_europe_in_data)].copy()

res_europe     = run_cs_ardl(df_europe,     label='Europe')
res_non_europe = run_cs_ardl(df_non_europe, label='OCDE hors Europe')

Pays EUROPÉENS reconnus dans le panel : ['Austria', 'Belgium', 'Czech Republic', 'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Netherlands', 'Norway', 'Poland', 'Portugal', 'Slovak Republic', 'Slovenia', 'Spain', 'Sweden', 'Switzerland', 'United Kingdom']

Pays OCDE hors Europe : ['Australia', 'Canada', 'Chile', 'Colombia', 'Costa Rica', 'Israel', 'Japan', 'Korea', 'Luxemburg', 'Mexico', 'New Zealand', 'Turkiye', 'United States']

  CS-ARDL(1,1) — Sous-échantillon : Europe
  Pays : 25  |  Observations : 672
                          PanelOLS Estimation Summary                           
Dep. Variable:                 growth   R-squared:                        0.6376
Estimator:                   PanelOLS   R-squared (Between):             -32.249
No. Observations:                 672   R-squared (Within):               0.6376
Date:                Sun, Mar 22 2026   R-squared (Overall):             -17.71

## B. Robustesse temporelle : avant et après la crise de 2008

On découpe le panel en deux sous-périodes :
- **Pré-crise** (1990–2007) : phase d'expansion du crédit et de croissance soutenue ;
- **Post-crise** (2008–2024) : phase de désendettement, taux bas, consolidation budgétaire.

> ⚠️ **Mise en garde** : la sous-période pré-crise compte environ 10–15 observations 
par pays selon la couverture de la base. Les résultats sont donc indicatifs — 
la puissance statistique est réduite et les CS-means sont calculées sur un T court.

In [21]:
df_pre  = df_master[df_master['year'] <= 2007].copy()
df_post = df_master[df_master['year'] >= 2008].copy()

print(f'Pré-crise  : {df_pre["year"].min()}–{df_pre["year"].max()}  '
      f'| {df_pre["country"].nunique()} pays, {len(df_pre)} obs')
print(f'Post-crise : {df_post["year"].min()}–{df_post["year"].max()}  '
      f'| {df_post["country"].nunique()} pays, {len(df_post)} obs')

res_pre  = run_cs_ardl(df_pre,  label=f'{df_pre["year"].min()}–2007')
res_post = run_cs_ardl(df_post, label=f'2008–{df_post["year"].max()}')

Pré-crise  : 1996–2007  | 38 pays, 456 obs
Post-crise : 2008–2024  | 38 pays, 646 obs

  CS-ARDL(1,1) — Sous-échantillon : 1996–2007
  Pays : 34  |  Observations : 334
                          PanelOLS Estimation Summary                           
Dep. Variable:                 growth   R-squared:                        0.3843
Estimator:                   PanelOLS   R-squared (Between):             -10.369
No. Observations:                 334   R-squared (Within):               0.3843
Date:                Sun, Mar 22 2026   R-squared (Overall):             -9.2311
Time:                        17:39:16   Log-likelihood                   -722.64
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      12.749
Entities:                          34   P-value                           0.0000
Avg Obs:                       9.8235   Distribution:                  F(14,286)
Min Obs:              

## C. Tableau comparatif des bêtas estimés

Le tableau ci-dessous synthétise les coefficients estimés pour chaque type de dette 
selon les différentes méthodes et sous-groupes.

**Lecture des colonnes :**
- `OLS` / `Effets Fixes` / `IV` : coefficient sur le lag-1 de la variable de dette ;
- `CS-ARDL CT` : effet de court terme ($\\beta_0$ sur $x_{it}$) ;
- `CS-ARDL LT` : effet de long terme, calculé comme $(\\beta_0+\\beta_1)/(1-\\phi)$ ;
- Colonnes de robustesse : effet CT sur chaque sous-groupe.

**Significativité :** \*\*\* $p<0.01$, \*\* $p<0.05$, \* $p<0.10$

In [22]:
x_vars_table = ['debt_hh', 'debt_corp', 'debt_gov']
labels_debt  = {
    'debt_hh':   'Dette des ménages',
    'debt_corp': 'Dette des entreprises',
    'debt_gov':  'Dette publique'
}

# --- Extraction des coefficients par méthode ---

# OLS (lag-1)
ols_coefs = {v: (model_ols.params.get(f'{v}_lag1', float('nan')),
                 get_pval(model_ols, f'{v}_lag1'))
             for v in x_vars_table}

# Effets Fixes (lag-1)
fe_coefs = {v: (results_fe.params.get(f'{v}_lag1', float('nan')),
                get_pval(results_fe, f'{v}_lag1'))
            for v in x_vars_table}

# IV 2SLS (debt_hh seulement)
iv_coefs = {
    'debt_hh':   (results_iv.params.get('debt_hh_lag1', float('nan')),
                  get_pval(results_iv, 'debt_hh_lag1')),
    'debt_corp': (float('nan'), float('nan')),
    'debt_gov':  (float('nan'), float('nan'))
}

# CS-ARDL complet
st_full, lt_full = extract_effects(res, x_vars_table, q=1)
pv_full = {v: get_pval(res, v) for v in x_vars_table}

# Robustesse
st_eu,    lt_eu    = extract_effects(res_europe,     x_vars_table)
st_noeu,  lt_noeu  = extract_effects(res_non_europe, x_vars_table)
st_pre,   lt_pre   = extract_effects(res_pre,        x_vars_table)
st_post,  lt_post  = extract_effects(res_post,       x_vars_table)

pv_eu   = {v: get_pval(res_europe,     v) for v in x_vars_table} if res_europe     else {v: float('nan') for v in x_vars_table}
pv_noeu = {v: get_pval(res_non_europe, v) for v in x_vars_table} if res_non_europe else {v: float('nan') for v in x_vars_table}
pv_pre  = {v: get_pval(res_pre,        v) for v in x_vars_table} if res_pre        else {v: float('nan') for v in x_vars_table}
pv_post = {v: get_pval(res_post,       v) for v in x_vars_table} if res_post       else {v: float('nan') for v in x_vars_table}

# --- Construction du DataFrame ---
rows = []
for v in x_vars_table:
    rows.append({
        'Type de dette'            : labels_debt[v],
        'OLS'                      : fmt(ols_coefs[v][0], ols_coefs[v][1]),
        'Effets Fixes'             : fmt(fe_coefs[v][0],  fe_coefs[v][1]),
        'IV (2SLS)'                : fmt(iv_coefs[v][0],  iv_coefs[v][1]),
        'CS-ARDL CT (full)'        : fmt(st_full[v],      pv_full[v]),
        'CS-ARDL LT (full)'        : fmt(lt_full[v]),
        'CS-ARDL CT — Europe'      : fmt(st_eu[v],        pv_eu[v]),
        'CS-ARDL CT — OCDE\u2216EU': fmt(st_noeu[v],      pv_noeu[v]),
        'CS-ARDL CT — Pré-crise'   : fmt(st_pre[v],       pv_pre[v]),
        'CS-ARDL CT — Post-crise'  : fmt(st_post[v],      pv_post[v]),
    })

comparison_table = pd.DataFrame(rows).set_index('Type de dette')

print('\n')
print('=' * 110)
print('  TABLEAU COMPARATIF — Bêta estimé (effet sur la croissance du PIB)')
print('  *** p<0.01  ** p<0.05  * p<0.10   |   — : non estimé')
print('=' * 110)
print(comparison_table.to_string())
print('=' * 110)
print("""
Notes :
  OLS / FE / IV   : coefficient sur le lag-1 de la variable de dette.
  CS-ARDL CT      : beta_0 sur x_it (effet contemporain).
  CS-ARDL LT      : (beta_0 + beta_1) / (1 - phi), effet de long terme.
  IV              : seule debt_hh est instrumentée (lag-2 comme instrument).
  Robustesse      : CS-means recalculées sur chaque sous-groupe.
  Sous-périodes   : T court (~10-15 obs/pays), résultats indicatifs.
""")

# Export CSV
comparison_table.to_csv('tableau_comparatif_betas.csv', encoding='utf-8-sig')
print('Tableau exporté : tableau_comparatif_betas.csv')



  TABLEAU COMPARATIF — Bêta estimé (effet sur la croissance du PIB)
  *** p<0.01  ** p<0.05  * p<0.10   |   — : non estimé
                              OLS Effets Fixes   IV (2SLS) CS-ARDL CT (full) CS-ARDL LT (full) CS-ARDL CT — Europe CS-ARDL CT — OCDE∖EU CS-ARDL CT — Pré-crise CS-ARDL CT — Post-crise
Type de dette                                                                                                                                                                         
Dette des ménages      -0.0533***   -0.0932***  -0.0487***        -0.3288***           -0.0109           -0.3695**            -0.2744**                 0.0709              -0.5905***
Dette des entreprises      0.0026      -0.0015           —            0.0030            0.0031              0.0227             -0.0609*              -0.0436**                 -0.0063
Dette publique         -0.0288***       0.0133           —        -0.1979***           -0.0352          -0.2135***            -0.1713**        

## Implémentation des effets de seuil

Dans la régression, au lieu d'avoir un seul $\beta \cdot x_{it}$, il y a désormais :$$\beta_{low} \cdot x_{it}^{low} + \beta_{high} \cdot x_{it}^{high} $$
Où :
- $x_{it}^{low}$ vaut la valeur de la dette si elle est $\le$ au seuil, et $0$ sinon.
- $x_{it}^{high}$ vaut la valeur de la dette si elle est $>$ au seuil, et $0$ sinon.

Cela permet en plus des effets de seuil d'analyser la variation de la croissance suite à une augmentation de la dette selon que l'on dépasse ou non le seuil.

In [23]:
# FIX : x_vars est explicitement défini ici pour que cette cellule
# soit auto-suffisante (ne dépende pas de l'état mémoire de la cellule CS-ARDL).
x_vars = CONFIG['interest_vars']  # ['debt_hh', 'debt_corp', 'debt_gov']

# =====================================================================
# ÉTAPE 1 — DÉTECTION ENDOGÈNE DES SEUILS (grille sur percentiles)
# =====================================================================
# Approche : pour chaque variable de dette, on balaye les percentiles
# de 15% à 85% (par pas de 5%) et on retient le seuil qui minimise
# la somme des carrés des résidus (SSR) d'une régression segmentée.
# C'est l'essence de la méthode de Hansen (1999), sans le test d'inférence
# formel (bootstrap) qui nécessiterait une librairie dédiée.
#
# La régression de référence pour le scan est une FE simple (within)
# avec les trois dettes demeaned par pays, plus rapide qu'un CS-ARDL
# complet à chaque itération.

from linearmodels.panel import PanelOLS as _PanelOLS
import warnings

def find_threshold_grid(df, var, other_vars, percentiles=range(15, 86, 5)):
    """
    Balaye les percentiles de 'var' et retourne le seuil minimisant le SSR
    d'une régression FE avec splitting de 'var' en deux régimes.
    """
    df_scan = df.dropna(subset=['growth', var] + other_vars).copy()
    df_scan = df_scan.set_index(['country', 'year'])

    best_ssr, best_thresh = np.inf, None
    ssr_by_thresh = {}

    for pct in percentiles:
        thresh = np.percentile(df_scan[var].values, pct)
        df_scan['_low']  = df_scan[var] * (df_scan[var] <= thresh).astype(float)
        df_scan['_high'] = df_scan[var] * (df_scan[var] >  thresh).astype(float)
        rhs_vars = ['_low', '_high'] + other_vars
        rhs = ' + '.join(rhs_vars)
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                m = _PanelOLS.from_formula(f'growth ~ {rhs} + EntityEffects',
                                           data=df_scan, drop_absorbed=True)
                r = m.fit()
                ssr = float(r.resids.pow(2).sum())
                ssr_by_thresh[thresh] = ssr
                if ssr < best_ssr:
                    best_ssr, best_thresh = ssr, thresh
        except Exception:
            continue

    return best_thresh, best_ssr, ssr_by_thresh


df_scan_base = df_master.dropna(subset=['growth'] + x_vars).copy()

print('Recherche des seuils endogènes (grille percentiles 15%–85%)...\n')
optimal_thresholds = {}
for var in x_vars:
    other = [v for v in x_vars if v != var]
    best_t, best_ssr, ssr_grid = find_threshold_grid(df_scan_base, var, other)
    optimal_thresholds[var] = best_t
    print(f"  {var:12s} → seuil optimal = {best_t:.1f}% du PIB  (SSR minimum = {best_ssr:.1f})")

print(f"\nSeuils retenus : {optimal_thresholds}")

# =====================================================================
# ÉTAPE 2 — RÉGRESSION CS-ARDL AVEC SEUILS ENDOGÈNES
# =====================================================================
thresholds = optimal_thresholds   # ← remplace les seuils exogènes

df_threshold = df_master.copy()

for var, limit in thresholds.items():
    dummy_name = f"dummy_low_{var}"
    df_threshold[dummy_name] = (df_threshold[var] <= limit).astype(int)
    df_threshold[f"{var}_low"]  = df_threshold[var] * df_threshold[dummy_name]
    df_threshold[f"{var}_high"] = df_threshold[var] * (1 - df_threshold[dummy_name])
    n_low  = df_threshold[dummy_name].sum()
    n_high = (1 - df_threshold[dummy_name]).sum()
    print(f"  {var:12s} : {n_low} obs sous le seuil, {n_high} au-dessus")

segmented_vars = []
for var in x_vars:
    segmented_vars.extend([f"{var}_low", f"{var}_high"])

p, q = 1, 1
df_cs = add_lags(df_threshold, "country", ["growth"] + segmented_vars, max_lag=max(p, q))

regressors = ["growth_lag1"] + segmented_vars
regressors += [f"{v}_lag1" for v in segmented_vars]

df_cs = add_cs_means(df_cs, "year", vars_list=regressors, prefix="cs_")
cs_regressors = [f"cs_{c}" for c in regressors]

needed = ["country", "year", "growth"] + regressors + cs_regressors
df_cs = df_cs[needed].dropna().set_index(["country", "year"])

rhs = " + ".join(regressors + cs_regressors)
model_threshold = PanelOLS.from_formula(f"growth ~ {rhs} + EntityEffects", data=df_cs)
res_threshold = model_threshold.fit(cov_type="clustered", cluster_entity=True)

print(res_threshold.summary)

Recherche des seuils endogènes (grille percentiles 15%–85%)...

  debt_hh      → seuil optimal = 39.3% du PIB  (SSR minimum = 22038.5)
  debt_corp    → seuil optimal = 61.3% du PIB  (SSR minimum = 21846.3)
  debt_gov     → seuil optimal = 90.2% du PIB  (SSR minimum = 21986.6)

Seuils retenus : {'debt_hh': 39.27, 'debt_corp': 61.28, 'debt_gov': 90.17}
  debt_hh      : 398 obs sous le seuil, 704 au-dessus
  debt_corp    : 299 obs sous le seuil, 803 au-dessus
  debt_gov     : 829 obs sous le seuil, 273 au-dessus
                          PanelOLS Estimation Summary                           
Dep. Variable:                 growth   R-squared:                        0.6102
Estimator:                   PanelOLS   R-squared (Between):             -11.009
No. Observations:                 945   R-squared (Within):               0.6102
Date:                Sun, Mar 22 2026   R-squared (Overall):             -7.7028
Time:                        17:39:18   Log-likelihood                   -2417.8